# Visualize Concept Generalization Tasks

Shows rules text (what the agent sees), entity observation (attributes), and oracle solution for each task instance.

| Task | Rule type | Description |
|------|-----------|-------------|
| C1 | Additive | `item_size = base(class) + modifier(role)` |
| C2 | Independent | `class→size, role→color` (independent dimensions) |
| C3 | Conditional | class selects a regime; regime controls which dimension role picks |
| C4 | Override | base rule `class→item`, overridden when `role=warrior` |

In [6]:
import os, sys

# Locate repo root regardless of where Jupyter was launched from.
# Works whether you open the notebook from the repo root or from eval/.
_cwd = os.path.abspath(os.getcwd())
_repo_root = _cwd if os.path.isdir(os.path.join(_cwd, 'tree_management')) \
             else os.path.dirname(_cwd)
os.chdir(_repo_root)
sys.path.insert(0, _repo_root)

from tree_management.registry import get_task
from env.env import AdventureEnv

def _get_gen_fn(task_type: str):
    return get_task(task_type).gen_fn

In [8]:
TASK_CONFIGS = {
    'prop_1': {
        'generator':   _get_gen_fn('prop_1'),
        'label':       'Prop 1  — Additive',
        'description': 'item_size = base(class) + modifier(role).  Both attrs contribute numerically to one property.',
    },
    'prop_2': {
        'generator':   _get_gen_fn('prop_2'),
        'label':       'Prop 2  — Independent',
        'description': 'class→size, role→color.  Each attr independently controls a different item dimension.',
    },
    'prop_3': {
        'generator':   _get_gen_fn('prop_3'),
        'label':       'Prop 3  — Conditional',
        'description': 'class selects a regime; regime determines which dimension role controls.',
    },
    'prop_4': {
        'generator':   _get_gen_fn('prop_4'),
        'label':       'Prop 4  — Override',
        'description': 'Base rule: class → item.  Exception: one role value overrides to a fixed item.',
    },
}

In [9]:
def task_card(task, split_label: str, idx: int, total: int, currency: int = 500) -> str:
    """Return a formatted string describing one task instance."""
    entity_name  = task.metadata.get("entity_instance", "?")
    attrs        = task.metadata.get("attributes", {})
    entity_obs   = f"{entity_name}  {attrs}"
    solution     = task.tree.get_solution()

    env = AdventureEnv([(task.tree, task.tree.root_id)], initial_currency=currency)
    rules_text = env.reset(
        tree_index=0,
        initial_currency=currency,
        rules_to_skip=task.rules_to_skip,
    )

    root_node   = task.tree.nodes[task.tree.root_id]
    root_action = root_node.meta.get('incoming_edge', '')
    goal        = f"{root_action} {root_node.argument}"

    lines = []
    bar = f"─── {split_label} {idx}/{total} " + "─" * max(0, 52 - len(split_label) - len(str(idx)) - len(str(total)))
    lines.append(bar)
    lines.append(f"  Entity : {entity_obs}")
    lines.append(f"  Goal   : {goal}")
    lines.append(f"  Hidden : {task.rules_to_skip}  (not shown in rules → must be inferred)")
    if task.demo_entity_names:
        lines.append(f"  Demos  : {task.demo_entity_names}")
    lines.append("")
    lines.append("  [Rules shown to agent]")
    for line in rules_text.strip().splitlines():
        lines.append(f"    {line}")
    lines.append("")
    lines.append("  [Oracle solution]")
    for step_str in solution:
        parts  = step_str.split(None, 1)
        action = parts[0]
        arg    = parts[1] if len(parts) > 1 else ''
        _, obs, _ = env.step(action, arg)
        lines.append(f"    > {step_str:<35}  ← {obs.message}")
    lines.append("")
    return "\n".join(lines)

In [4]:
def visualize(task_type: str = 'c3', use_nonce: bool = False,
              max_source: int = 3, max_gen: int = 3, currency: int = 500):
    cfg = TASK_CONFIGS[task_type]
    print()
    print('━' * 60)
    print(f"  {cfg['label']}")
    for line in cfg['description'].splitlines():
        print(f"  {line}")
    print(f"  (nonce={use_nonce})")
    print('━' * 60)

    source_tasks = cfg['generator'](cfg['elements'], split='source', use_nonce=use_nonce)
    gen_tasks    = cfg['generator'](cfg['elements'], split='gen',    use_nonce=use_nonce)

    print(f"\n  SOURCE split: {len(source_tasks)} tasks total  (showing {min(max_source, len(source_tasks))})\n")
    for i, task in enumerate(source_tasks[:max_source], 1):
        print(task_card(task, 'SOURCE', i, len(source_tasks), currency))

    print(f"\n  GEN split: {len(gen_tasks)} tasks total  (showing {min(max_gen, len(gen_tasks))})\n")
    print("  NOTE: In evaluation the agent sees source demos but not a demo for the gen entity.")
    print("        The concept must be applied to a (class, role) combo not seen in source.\n")
    for i, task in enumerate(gen_tasks[:max_gen], 1):
        print(task_card(task, 'GEN', i, len(gen_tasks), currency))

## C1 — Additive

In [12]:
visualize('c1', use_nonce=True, max_source=2, max_gen=2)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Concept C1  — 2-attr additive
  item_size = base(class) + modifier(role).  Both attrs contribute numerically to one property.
  (nonce=True)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  SOURCE split: 6 tasks total  (showing 2)

─── SOURCE 1/6 ────────────────────────────────────────────
  Entity : Drev_Kornak — vrel: vrel_A, korv: korv_X
  Goal   : defeat Drev_Kornak
  Hidden : ['blorn_fang']  (not shown in rules → must be inferred)

  [Rules shown to agent]
    Rules in the world:
    defeat Drev_Kornak requires: (1) must be at citadel
    
    Your final goal:
    defeat Drev_Kornak

  [Oracle solution]
    > go forge                             ← Traveled to forge. You are now at forge.
    > buy blorn_fang                       ← Bought blorn_fang for 60 currency (remaining: 440). You now have blorn_fang.
    > go citadel                           ← Traveled to citadel. You are now at citadel.
    > 

## C2 — Compositional

In [ ]:
visualize('c2', use_nonce=False, max_source=2, max_gen=2)

## C3 — Conditional

In [ ]:
visualize('c3', use_nonce=False, max_source=3, max_gen=2)

## C4 — Override / Exception

In [ ]:
visualize('c4', use_nonce=False, max_source=3, max_gen=2)

## Nonce-word variants
Run any cell above with `use_nonce=True`, or use the cell below to compare.

In [ ]:
visualize('c3', use_nonce=True, max_source=2, max_gen=1)